In [ ]:
%load_ext autoreload
%autoreload 2
from multistyleseg.experiments.fundus.ensemble import get_ensemble_model
from nntools.utils import Config
from multistyleseg.data.fundus.factory import ALL_DATASETS, get_datamodule_from_config
from multistyleseg.data.fundus.consts import ALL_CLASSES

from multistyleseg.models.factory import ModelType, get_model
from multistyleseg.trainers.fundus import FundusSegmentationTrainer
from multistyleseg.utils import CKPT_DIR
import pandas as pd
from multistyleseg.analysis.evaluation_toolkit import (
    evaluate_model_pixel,
    evaluate_model_detection,
    aggregate_detection_results,
    detection_size_distributions,
)
from pathlib import Path

In [2]:
config = Config("/home/clement/Documents/Projets/MultiStyleSeg/configs/fundus.yaml")
config["data"]["eval_batch_size"] = 16
config["data"]["batch_size"] = 16

datamodule = get_datamodule_from_config(
    config["datasets"], ALL_DATASETS, config["data"]
)

test_dataloaders = datamodule.test_dataloader()

ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.


Training on datasets: [<FundusDataset.IDRID: 'IDRID'>, <FundusDataset.FGADR: 'FGADR'>, <FundusDataset.MESSIDOR: 'MESSIDOR'>, <FundusDataset.DDR: 'DDR'>, <FundusDataset.RETLES: 'RETLES'>]
Training samples: 2883, 361 batches
Validation samples: 588, 74 batches
Test samples: 5


In [3]:
ensemble_model = get_ensemble_model(1024).cuda()
ensemble_model = ensemble_model.eval()


In [4]:
OUTPUT_FOLDER = Path("Results")
OUTPUT_FOLDER.mkdir(exist_ok=True)

In [5]:
NUM_CLASSES = 5
pixel_frames = {}
detection_frames = {}

ckpt_folder = CKPT_DIR.resolve()
for model_path in ckpt_folder.rglob("*.ckpt"):
    model_type = model_path.parent.name.split("fundus_")[1]
    arch = get_model(
        model_type=ModelType(model_type),
        in_channels=3,
        out_channels=5,
        img_size=1024,
    )
    model = FundusSegmentationTrainer.load_from_checkpoint(
        checkpoint_path=model_path,
        model=arch,
    )
    path_pixel = OUTPUT_FOLDER / f"fundus_{model_type}_pixel.pkl"
    path_det = OUTPUT_FOLDER / f"fundus_{model_type}_detection.pkl"
    if path_pixel.exists() and path_det.exists():
        print(f"Results for {model_type} already exist, skipping...")
        pixel_frames[model_type] = pd.read_pickle(path_pixel)
        detection_frames[model_type] = pd.read_pickle(path_det)
        continue

    # ---- Pixel metrics (same as before, but cleaner) ----
    df_pixel = evaluate_model_pixel(
        model,
        test_dataloaders,
        ALL_CLASSES,
        num_classes=NUM_CLASSES,
    )
    df_pixel.to_pickle(OUTPUT_FOLDER / f"fundus_{model_type}_pixel.pkl")
    pixel_frames[model_type] = df_pixel

    # ---- Detection metrics (new) ----
    det_results = evaluate_model_detection(
        model,
        test_dataloaders,
        ALL_CLASSES,
        num_classes=NUM_CLASSES,
        iou_threshold=0.10,  # any overlap > 10% counts as a match
        min_blob_area=0,  # set >0 to ignore dust-sized components
    )
    df_det = aggregate_detection_results(det_results, ALL_CLASSES)
    df_det.to_pickle(OUTPUT_FOLDER / f"fundus_{model_type}_detection.pkl")
    detection_frames[model_type] = df_det

    # ---- Size distributions for plots ----
    df_sizes = detection_size_distributions(det_results, ALL_CLASSES)
    df_sizes.to_pickle(OUTPUT_FOLDER / f"fundus_{model_type}_sizes.pkl")

Eval on IDRID:   0%|          | 0/4 [00:00<?, ?it/s]

No positive samples in targets, true positive value should be meaningless. Returning zero tensor in true positive score


Eval on FGADR:   0%|          | 0/47 [00:00<?, ?it/s]

Eval on MESSIDOR:   0%|          | 0/8 [00:00<?, ?it/s]

Eval on DDR:   0%|          | 0/29 [00:00<?, ?it/s]

...

Detection eval on FGADR:   0%|          | 0/47 [00:00<?, ?it/s]

Detection eval on MESSIDOR:   0%|          | 0/8 [00:00<?, ?it/s]

Detection eval on DDR:   0%|          | 0/29 [00:00<?, ?it/s]

Corrupt JPEG data: 40 extraneous bytes before marker 0xd9
Corrupt JPEG data: 40 extraneous bytes before marker 0xd9


Detection eval on RETLES:   0%|          | 0/40 [00:00<?, ?it/s]

In [7]:
NUM_CLASSES = 5
pixel_frames = {}
detection_frames = {}
ENSEMBLE_DONE = False
model_type = "ensemble"
model = ensemble_model
path_pixel = OUTPUT_FOLDER / f"fundus_{model_type}_pixel.pkl"
path_det = OUTPUT_FOLDER / f"fundus_{model_type}_detection.pkl"
if path_pixel.exists() and path_det.exists():
    print(f"Results for {model_type} already exist, skipping...")
    pixel_frames[model_type] = pd.read_pickle(path_pixel)
    detection_frames[model_type] = pd.read_pickle(path_det)
    ENSEMBLE_DONE = True

if not ENSEMBLE_DONE:
    # ---- Pixel metrics (same as before, but cleaner) ----
    df_pixel = evaluate_model_pixel(
        model,
        test_dataloaders,
        ALL_CLASSES,
        num_classes=NUM_CLASSES,
    )
    df_pixel.to_pickle(OUTPUT_FOLDER / f"fundus_{model_type}_pixel.pkl")
    pixel_frames[model_type] = df_pixel

    # ---- Detection metrics (new) ----
    det_results = evaluate_model_detection(
        model,
        test_dataloaders,
        ALL_CLASSES,
        num_classes=NUM_CLASSES,
        iou_threshold=0.10,  # any overlap > 10% counts as a match
        min_blob_area=0,  # set >0 to ignore dust-sized components
    )
    df_det = aggregate_detection_results(det_results, ALL_CLASSES)
    df_det.to_pickle(OUTPUT_FOLDER / f"fundus_{model_type}_detection.pkl")
    detection_frames[model_type] = df_det

    # ---- Size distributions for plots ----
    df_sizes = detection_size_distributions(det_results, ALL_CLASSES)
    df_sizes.to_pickle(OUTPUT_FOLDER / f"fundus_{model_type}_sizes.pkl")

Eval on IDRID:   0%|          | 0/4 [00:00<?, ?it/s]

Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:345.)
Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:345.)
No positive samples in targets, true positive value should be meaningless. Returning zero tensor in true positive score


Eval on FGADR:   0%|          | 0/47 [00:00<?, ?it/s]

Eval on MESSIDOR:   0%|          | 0/8 [00:00<?, ?it/s]

Eval on DDR:   0%|          | 0/29 [00:00<?, ?it/s]

...

Detection eval on FGADR:   0%|          | 0/47 [00:00<?, ?it/s]

Detection eval on MESSIDOR:   0%|          | 0/8 [00:00<?, ?it/s]

Detection eval on DDR:   0%|          | 0/29 [00:00<?, ?it/s]

Corrupt JPEG data: 40 extraneous bytes before marker 0xd9
Corrupt JPEG data: 40 extraneous bytes before marker 0xd9


Detection eval on RETLES:   0%|          | 0/40 [00:00<?, ?it/s]

In [ ]:
ensemble_model = get_ensemble_model(
    1024, model_choices=["UNET", "SEGFORMER", "SERESNET_UNET"]
).cuda()
ensemble_model = ensemble_model.eval()
NUM_CLASSES = 5
pixel_frames = {}
detection_frames = {}
ENSEMBLE_DONE = False
model_type = "ensemble_best"
model = ensemble_model
path_pixel = OUTPUT_FOLDER / f"fundus_{model_type}_pixel.pkl"
path_det = OUTPUT_FOLDER / f"fundus_{model_type}_detection.pkl"
if path_pixel.exists() and path_det.exists():
    print(f"Results for {model_type} already exist, skipping...")
    pixel_frames[model_type] = pd.read_pickle(path_pixel)
    detection_frames[model_type] = pd.read_pickle(path_det)
    ENSEMBLE_DONE = True

if not ENSEMBLE_DONE:
    # ---- Pixel metrics (same as before, but cleaner) ----
    df_pixel = evaluate_model_pixel(
        model,
        test_dataloaders,
        ALL_CLASSES,
        num_classes=NUM_CLASSES,
    )
    df_pixel.to_pickle(OUTPUT_FOLDER / f"fundus_{model_type}_pixel.pkl")
    pixel_frames[model_type] = df_pixel

    # ---- Detection metrics (new) ----
    det_results = evaluate_model_detection(
        model,
        test_dataloaders,
        ALL_CLASSES,
        num_classes=NUM_CLASSES,
        iou_threshold=0.10,  # any overlap > 10% counts as a match
        min_blob_area=0,  # set >0 to ignore dust-sized components
    )
    df_det = aggregate_detection_results(det_results, ALL_CLASSES)
    df_det.to_pickle(OUTPUT_FOLDER / f"fundus_{model_type}_detection.pkl")
    detection_frames[model_type] = df_det

    # ---- Size distributions for plots ----
    df_sizes = detection_size_distributions(det_results, ALL_CLASSES)
    df_sizes.to_pickle(OUTPUT_FOLDER / f"fundus_{model_type}_sizes.pkl")

Eval on IDRID:   0%|          | 0/4 [00:00<?, ?it/s]

Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:345.)
Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:345.)
No positive samples in targets, true positive value should be meaningless. Returning zero tensor in true positive score


Eval on FGADR:   0%|          | 0/47 [00:00<?, ?it/s]

Eval on MESSIDOR:   0%|          | 0/8 [00:00<?, ?it/s]

Eval on DDR:   0%|          | 0/29 [00:00<?, ?it/s]

...

Detection eval on FGADR:   0%|          | 0/47 [00:00<?, ?it/s]

Detection eval on MESSIDOR:   0%|          | 0/8 [00:00<?, ?it/s]

Detection eval on DDR:   0%|          | 0/29 [00:00<?, ?it/s]

Corrupt JPEG data: 40 extraneous bytes before marker 0xd9
Corrupt JPEG data: 40 extraneous bytes before marker 0xd9


Detection eval on RETLES:   0%|          | 0/40 [00:00<?, ?it/s]